# 03 - Feature Engineering: Walmart Weekly Sales

This notebook creates additional features to support forecasting and modelling.

**Main goals:**
- Load the cleaned weekly sales data.
- Create calendar-based features (year, month, week of year, etc.).
- Generate lagged sales features to capture short term patterns.
- Generate rolling statistics (moving averages and volatility).
- Save a processed dataset for use in modelling.


In [1]:
# Load cleaned dataset and import libraries
import pandas as pd
import numpy as np
from pathlib import Path

# Define project root as parent of the notebooks folder
PROJECT_ROOT = Path.cwd().resolve().parent

# Define paths for cleaned and processed data
DATA_CLEAN = PROJECT_ROOT / "data" / "clean"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

# Ensure processed data folder exists
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Path to cleaned dataset created in Notebook 1
clean_path = DATA_CLEAN / "walmart_sales_clean.csv"

# Load cleaned dataset, parsing 'date' as datetime
df = pd.read_csv(clean_path, parse_dates=["date"])

print("Data loaded from:", clean_path)
print("Shape:", df.shape)
df.head()


Data loaded from: C:\Users\Mist\Documents\Portfolio\P1. Retail Sales Forecasting and Stock Optimisation\data\clean\walmart_sales_clean.csv
Shape: (6435, 8)


,store,date,weekly_sales,holiday_flag,temperature,fuel_price,cpi,unemployment
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106


## Ensure data is sorted by store and date

Many time based features (lags and rolling windows) rely on data being in the correct order.

Here we:
- Sort by `store` and `date`.
- Reset the index for a clean, sequential row ordering.


In [2]:
# Sort the data to ensure correct temporal order within each store
df = df.sort_values(["store", "date"]).reset_index(drop=True)

# Quick check of sorted data
df.head()


,store,date,weekly_sales,holiday_flag,temperature,fuel_price,cpi,unemployment
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106


## Create calendar-based features

To capture seasonality and temporal effects, we derive calendar features from `date`:

- `year`
- `month`
- `weekofyear` (ISO week)
- `dayofweek` (even though data is weekly, this is kept for completeness)
- `is_month_start`
- `is_month_end`

These features are commonly used in time series models and tree-based models.


In [3]:
# Create year, month, week of year, and day of week features
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)  # ISO week number
df["dayofweek"] = df["date"].dt.dayofweek  # 0 = Monday, 6 = Sunday

# Flags for month boundaries (might be useful for certain models)
df["is_month_start"] = df["date"].dt.is_month_start.astype(int)
df["is_month_end"] = df["date"].dt.is_month_end.astype(int)

# Inspect the new columns
df[["date", "year", "month", "weekofyear", "dayofweek", "is_month_start", "is_month_end"]].head()


,date,year,month,weekofyear,dayofweek,is_month_start,is_month_end
0,2010-02-05,2010,2,5,4,0,0
1,2010-02-12,2010,2,6,4,0,0
2,2010-02-19,2010,2,7,4,0,0
3,2010-02-26,2010,2,8,4,0,0
4,2010-03-05,2010,3,9,4,0,0


## Create lagged sales features

Lag features help the model learn from recent past values.

For each store, we create lags of `weekly_sales`:
- 1 week ago (`lag_1`)
- 2 weeks ago (`lag_2`)
- 4 weeks ago (`lag_4`)

We use a groupby on `store` to ensure lags are calculated within each store's time series.


In [4]:
# Define the lags (in weeks) to create for weekly_sales
lag_list = [1, 2, 4]

# For each store, shift weekly_sales by the specified number of weeks
for lag in lag_list:
    col_name = f"weekly_sales_lag_{lag}"
    # Group by store to prevent leakage across stores
    df[col_name] = df.groupby("store")["weekly_sales"].shift(lag)

# Inspect a few rows to verify lag creation
df[["store", "date", "weekly_sales"] + [f"weekly_sales_lag_{l}" for l in lag_list]].head(10)


,store,date,weekly_sales,weekly_sales_lag_1,weekly_sales_lag_2,weekly_sales_lag_4
0,1,2010-02-05,1643690.90,NaN,NaN,NaN
1,1,2010-02-12,1641957.44,1643690.90,NaN,NaN
2,1,2010-02-19,1611968.17,1641957.44,1643690.90,NaN
3,1,2010-02-26,1409727.59,1611968.17,1641957.44,NaN
4,1,2010-03-05,1554806.68,1409727.59,1611968.17,1643690.90
5,1,2010-03-12,1439541.59,1554806.68,1409727.59,1641957.44
6,1,2010-03-19,1472515.79,1439541.59,1554806.68,1611968.17
7,1,2010-03-26,1404429.92,1472515.79,1439541.59,1409727.59
8,1,2010-04-02,1594968.28,1404429.92,1472515.79,1554806.68
9,1,2010-04-09,1545418.53,1594968.28,1404429.92,1439541.59


## Create rolling window features

Rolling statistics summarise recent history and can stabilise noisy series.

For each store, we calculate over a 4-week window:

- `rolling_mean_4`: mean weekly sales over the last 4 weeks
- `rolling_std_4`: standard deviation of weekly sales over the last 4 weeks

These features can help models understand local level and volatility.


In [5]:
# Define the rolling window size (in weeks)
window_size = 4

# Group by store and compute rolling statistics for weekly_sales
df["rolling_mean_4"] = (
    df.groupby("store")["weekly_sales"]
    .transform(lambda x: x.rolling(window=window_size, min_periods=1).mean())
)

df["rolling_std_4"] = (
    df.groupby("store")["weekly_sales"]
    .transform(lambda x: x.rolling(window=window_size, min_periods=1).std())
)

# Inspect a subset of columns to verify rolling features
df[["store", "date", "weekly_sales", "rolling_mean_4", "rolling_std_4"]].head(10)


,store,date,weekly_sales,rolling_mean_4,rolling_std_4
0,1,2010-02-05,1643690.90,1.643691e+06,NaN
1,1,2010-02-12,1641957.44,1.642824e+06,1225.741321
2,1,2010-02-19,1611968.17,1.632539e+06,17835.791719
3,1,2010-02-26,1409727.59,1.576836e+06,112353.415114
4,1,2010-03-05,1554806.68,1.554615e+06,103135.002548
5,1,2010-03-12,1439541.59,1.504011e+06,95360.050839
6,1,2010-03-19,1472515.79,1.469148e+06,62599.457150
7,1,2010-03-26,1404429.92,1.467823e+06,64308.381016
8,1,2010-04-02,1594968.28,1.477864e+06,82871.762296
9,1,2010-04-09,1545418.53,1.504333e+06,83458.043354


## Check for missing values after feature engineering

Lag features will naturally create missing values at the start of each store's history.

Here we:
- Count missing values.
- Decide whether we want to keep these rows or drop early rows with insufficient history.


In [6]:
# Count missing values in each column
missing_after_fe = df.isna().sum().sort_values(ascending=False)
missing_after_fe


weekly_sales_lag_4    180
weekly_sales_lag_2     90
rolling_std_4          45
weekly_sales_lag_1     45
weekofyear              0
rolling_mean_4          0
is_month_end            0
is_month_start          0
dayofweek               0
store                   0
date                    0
year                    0
unemployment            0
cpi                     0
fuel_price              0
temperature             0
holiday_flag            0
weekly_sales            0
month                   0
dtype: int64

## Prepare model-ready dataset (only remove required missing lag values)

Lag features naturally produce missing values at the beginning of each store's
time series. This is expected.

Instead of removing all rows with *any* missing lag values, we take a more
professional approach:

- We keep the full feature dataset (`df`) for EDA and diagnostics.
- We create a **model-ready dataset** (`df_model_ready`) where:
  - We only drop rows missing the largest lag (`weekly_sales_lag_4`)
  - This ensures each record has at least 4 full weeks of history
  - We avoid removing too many rows unnecessarily


In [7]:
# Create a model-ready dataset by removing only rows missing lag_4,
# because lag_4 requires the most history and ensures consistency.

# Count rows before filtering
before_rows = df.shape[0]

# Drop only rows where lag_4 is missing
df_model_ready = df.dropna(subset=["weekly_sales_lag_4"]).copy()

# Count rows after filtering
after_rows = df_model_ready.shape[0]

print(f"Rows before: {before_rows}")
print(f"Rows after:  {after_rows}")
print(f"Rows removed due to insufficient history: {before_rows - after_rows}")


Rows before: 6435
Rows after:  6255
Rows removed due to insufficient history: 180


## Save feature engineered dataset

We now save:
- The full feature engineered dataset (`df`)
- The model ready dataset with complete lag features (`df_model_ready`)

These will be used in the modelling notebook.


In [8]:
# Paths for saving processed datasets
full_features_path = DATA_PROCESSED / "walmart_features_full.csv"
model_ready_path = DATA_PROCESSED / "walmart_features_model_ready.csv"

# Save DataFrames to CSV
df.to_csv(full_features_path, index=False)
df_model_ready.to_csv(model_ready_path, index=False)

print("Saved feature engineered datasets:")
print("-", full_features_path)
print("-", model_ready_path)
print("Full features shape:", df.shape)
print("Model ready shape:", df_model_ready.shape)


Saved feature engineered datasets:
- C:\Users\Mist\Documents\Portfolio\P1. Retail Sales Forecasting and Stock Optimisation\data\processed\walmart_features_full.csv
- C:\Users\Mist\Documents\Portfolio\P1. Retail Sales Forecasting and Stock Optimisation\data\processed\walmart_features_model_ready.csv
Full features shape: (6435, 19)
Model ready shape: (6255, 19)
